# F-08: EC-tower vs external (SMS/MET) sensor sourcing for the gap-filling drivers (D-35)

Two data variants under one identical harness (full-period gap-CV, all towers):
- **`EC`** baseline: `consolidated_hourly.csv` + `reddyproc_processed.csv`; `ts = TS_1_1_1 [Tower 9]`.
- **`EXT`** external: `consolidated_hourly_SMS_MET.csv` + `reddyproc_processed_SMS_MET.csv`; SWIN/TA/WS←Site, `ts`←per-catchment external.

Metrics: median **R² / RMSE / MAE / MBE** per gap scenario. MDS / RFm-solo / RFm-pooled per tower. See `F08_results.md`.

In [1]:
from pathlib import Path
import datetime, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
PLAUS_LOW,PLAUS_HIGH=-500,3000
N_REPS,MASK_FRAC=5,0.25
SCENARIOS={"vs":1,"s":4,"m":32,"l":288,"m1":"mixed"}
LSU={"cattle":1.0,"sheep":0.1,"lamb":0.05}; AUX=["_hs","_hc","_ds","_dc"]; LAG_HOURS=[168,336,504,672]
AREA={2:6.65,4:7.75,9:7.75}; C4="Catchment 4 After  2013/08/13"
DOMAIN={2:("2017-10-01","2019-06-30"),4:("2017-10-01","2023-12-31"),9:("2020-02-01","2023-12-31")}

## 1  Per-tower config (TS source switches by variant)

In [2]:
def ts_col_for(t, variant):
    if variant=="EC": return "TS_1_1_1 [Tower 9]"
    cat=C4 if t==4 else f"Catchment {t}"; return f"Soil Temperature @ 15cm Depth (oC) [{cat}]"

def cfg(t, ts_col):
    cat=C4 if t==4 else f"Catchment {t}"
    met=[f"SWIN_1_1_1 [Tower {t}]",f"TA_0_0_1 [Tower {t}]",f"VPD_0_0_1 [Tower {t}]",f"PPFD_1_1_1 [Tower {t}]",
         f"RN_1_1_1 [Tower {t}]",f"WS_0_0_1 [Tower {t}]",f"USTAR_0_0_1 [Tower {t}]",f"SHF_1_1_1 [Tower {t}]",
         f"Precipitation (mm) [{cat}]",ts_col,f"Soil Moisture @ 10cm Depth (%) [{cat}]"]
    return dict(t=t,cat=cat,area=AREA[t],tgt=f"FCH4_1_1_1 [Tower {t}]",ssitc=f"FCH4_SSITC_TEST_1_1_1 [Tower {t}]",
        sw=f"SWIN_1_1_1 [Tower {t}]",ta=f"TA_0_0_1 [Tower {t}]",met=met,fc=f"FC_1_1_1 [Tower {t}]",
        gpp=f"GPP [Tower {t}]",reco=f"Reco [Tower {t}]",swc=f"Soil Moisture @ 10cm Depth (%) [{cat}]",ts=ts_col,
        liv={s:f"{s}_{cat}" for s in LSU},mc=f"mgmt_t{t}_cut_recency",mm=f"mgmt_t{t}_manure_recency")

def feat_for(variant):
    tc=ts_col_for(2,variant)
    return ([m.split(" [")[0] for m in cfg(2,tc)["met"]]+["fc"]+AUX+["lsu_dens","graze"]
            +[f"swc_l{l}" for l in LAG_HOURS]+[f"ts_l{l}" for l in LAG_HOURS]+["mgmt_cut","mgmt_manure","gpp","reco"])
DUM=["is_t2","is_t4","is_t9"]

## 2  Load both data variants

In [3]:
def load_variant(cons, proc):
    d=pd.read_csv(HOURLY/cons,low_memory=False); d["Datetime"]=pd.to_datetime(d["Datetime"],format="mixed"); d=d.set_index("Datetime")
    for f in ["fco2_gapfilled.csv","management_features.csv",proc]:
        e=pd.read_csv(HOURLY/f,low_memory=False); e["Datetime"]=pd.to_datetime(e["Datetime"],format="mixed")
        d=d.join(e.set_index("Datetime"),how="left")
    for t in [2,4,9]: d[f"FC_1_1_1 [Tower {t}]"]=d[f"FC_gapfilled [Tower {t}]"]
    return d
DF={"EC": load_variant("consolidated_hourly.csv","reddyproc_processed.csv"),
    "EXT":load_variant("consolidated_hourly_SMS_MET.csv","reddyproc_processed_SMS_MET.csv")}
print({k:v.shape for k,v in DF.items()})

{'EC': (70153, 522), 'EXT': (70153, 524)}


## 3  Functions (calendar gaps, MDS, feature frame, metrics)

In [4]:
def insert_calendar_gaps(df_qc, target, domain_mask, gap_hours, n_reps=N_REPS, seed=0):
    dom_ts=df_qc.index[domain_mask]; valid=df_qc.loc[domain_mask,target].notna().values
    n=len(dom_ts); target_n=max(1,int(valid.sum()*MASK_FRAC)); rb=np.random.default_rng(seed); reps=[]
    for _ in range(n_reps):
        rng=np.random.default_rng(int(rb.integers(0,2**31))); occ=np.zeros(n,bool); m=0
        for sp in rng.permutation(n):
            if m>=target_n: break
            gh=int(rng.choice([1,4,32,288])) if gap_hours=="mixed" else gap_hours; ep=min(int(sp)+gh,n)
            if occ[sp:ep].any(): continue
            occ[sp:ep]=True; m+=int(valid[sp:ep].sum())
        reps.append(dom_ts[occ & valid])
    return reps

def mds_fill_batch(df_obs, target, sw_col, ta_col, gap_ts):
    SW_TOL,TA_TOL=50.0,2.5; WINDOWS=[pd.Timedelta(days=d) for d in [7,14,28,91]]
    av=df_obs[df_obs[target].notna()]; ay=av[target].values.astype(float)
    ahr=av.index.hour.to_numpy(); adoy=av.index.dayofyear.to_numpy(); ats=av.index.to_numpy()
    asw=av[sw_col].values.astype(float); ata=av[ta_col].values.astype(float)
    gi=pd.DatetimeIndex(gap_ts); gsw=df_obs.reindex(gi)[sw_col].values.astype(float); gta=df_obs.reindex(gi)[ta_col].values.astype(float)
    preds={}
    for i,t in enumerate(gap_ts):
        tt=np.datetime64(t); hr=t.hour; doy=t.dayofyear; sv=gsw[i]; tv=gta[i]
        day=(not np.isnan(sv)) and sv>10.0; filled=False
        for wd in WINDOWS:
            w=wd.to_timedelta64(); m=(ats>=tt-w)&(ats<=tt+w)&(np.abs(ahr-hr)<=1)
            if not np.isnan(tv): m&=(np.abs(ata-tv)<=TA_TOL)|np.isnan(ata)
            if day and not np.isnan(sv): m&=(np.abs(asw-sv)<=SW_TOL)|np.isnan(asw)
            c=ay[m]
            if len(c)>=1: preds[t]=float(np.nanmean(c)); filled=True; break
        if not filled:
            sh=np.abs(ahr-hr)<=1; dd=np.minimum(np.abs(adoy-doy),365-np.abs(adoy-doy)); c=ay[sh&(dd<=7)]
            if len(c)>=1: preds[t]=float(np.nanmean(c))
    return preds

def frame(t, pooled, d, ts_col):
    c=cfg(t, ts_col); d=d.copy(); tgt=c["tgt"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna()&~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    h,doy=d.index.hour,d.index.dayofyear
    d["_hs"]=np.sin(2*np.pi*h/24);d["_hc"]=np.cos(2*np.pi*h/24);d["_ds"]=np.sin(2*np.pi*doy/365);d["_dc"]=np.cos(2*np.pi*doy/365)
    g=pd.DataFrame(index=d.index); g["target"]=d[tgt]
    for k in c["met"]:
        nm=k.split(" [")[0]; g[nm]= d[k+"__f"] if (k+"__f") in d.columns else d[k]
    g["fc"]=d[c["fc"]]
    for a in AUX: g[a]=d[a]
    lsu=sum(d[c["liv"][s]].fillna(0)*w for s,w in LSU.items())
    g["lsu_dens"]=lsu/c["area"]; g["graze"]=(sum(d[c["liv"][s]].fillna(0) for s in LSU)>0).astype(float)
    swc=d[c["swc"]+"__f"] if (c["swc"]+"__f") in d.columns else d[c["swc"]]
    ts=d[ts_col+"__f"] if (ts_col+"__f") in d.columns else d[ts_col]
    for lag in LAG_HOURS: g[f"swc_l{lag}"]=swc.shift(lag); g[f"ts_l{lag}"]=ts.shift(lag)
    g["mgmt_cut"]=d[c["mc"]]; g["mgmt_manure"]=d[c["mm"]]; g["gpp"]=d[c["gpp"]]; g["reco"]=d[c["reco"]]
    if pooled:
        for tt in [2,4,9]: g[f"is_t{tt}"]=1.0 if tt==t else 0.0
    g["_y"]=d.index.year
    return g

def mets(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r2=r2_score(y,p) if np.var(y)>0 else np.nan
    return r2, float(np.sqrt(np.mean((p-y)**2))), float(np.mean(np.abs(p-y))), float(np.mean(p-y))
def med_metrics(rows):
    if not rows: return {k:np.nan for k in ["R2","RMSE","MAE","MBE"]}
    a=np.array(rows,float)
    return {"R2":np.nanmedian(a[:,0]),"RMSE":np.median(a[:,1]),"MAE":np.median(a[:,2]),"MBE":np.median(a[:,3])}

## 4  Run (median R²/RMSE/MAE/MBE per scenario), full-period gap-CV all towers

In [5]:
def fit(feat,trd):
    imp=SimpleImputer(strategy="mean"); rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(trd[feat].values),trd["target"].values); return rf,imp
def dom_mask(idx,t):
    a,b=DOMAIN[t]; return (idx>=pd.Timestamp(a))&(idx<=pd.Timestamp(b))

def run_mds(t, variant):
    d=DF[variant].copy(); c=cfg(t, ts_col_for(t,variant)); tgt=c["tgt"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna()&~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    dm=dom_mask(d.index,t); out={}
    for sc,gh in SCENARIOS.items():
        rr=[]
        for gt in insert_calendar_gaps(d,tgt,dm,gh):
            if len(gt)<5: continue
            sav=d.loc[gt,tgt].copy(); d.loc[gt,tgt]=np.nan
            p=mds_fill_batch(d,tgt,c["sw"],c["ta"],list(gt)); d.loc[gt,tgt]=sav
            ts=[x for x in gt if x in p]
            if len(ts)>=5: rr.append(mets(sav.loc[ts].values,np.array([p[x] for x in ts])))
        out[sc]=med_metrics(rr)
    return out

def run_rf(t, pooled, variant):
    feat=feat_for(variant)+(DUM if pooled else [])
    towers=[2,4,9] if pooled else [t]
    frames={tt:frame(tt, pooled, DF[variant], ts_col_for(tt,variant)) for tt in towers}
    g=frames[t]; idx=g.index; out={}; dm=dom_mask(idx,t)
    for sc,gh in SCENARIOS.items():
        rr=[]
        for gt in insert_calendar_gaps(g,"target",dm,gh):
            if len(gt)<5: continue
            base=g[dm & g["target"].notna().values]; trd=base.drop(index=gt,errors="ignore")
            if pooled:
                others=[]
                for tt in towers:
                    if tt==t: continue
                    gg=frames[tt]; dmt=dom_mask(gg.index,tt); others.append(gg[dmt & gg["target"].notna().values])
                trd=pd.concat([trd]+others,ignore_index=True)
            rf,imp=fit(feat,trd); yp=rf.predict(imp.transform(g.loc[gt,feat].values))
            rr.append(mets(g.loc[gt,"target"].values,yp))
        out[sc]=med_metrics(rr)
    return out

RESU={}
for variant in ["EC","EXT"]:
    for t in [2,4,9]:
        RESU[(variant,t,"MDS")]=run_mds(t,variant)
        RESU[(variant,t,"RFm_solo")]=run_rf(t,False,variant)
        RESU[(variant,t,"RFm_pool")]=run_rf(t,True,variant)
        print("done",variant,"Tower",t,flush=True)
print("ALL DONE")

done EC Tower 2


done EC Tower 4


done EC Tower 9


done EXT Tower 2


done EXT Tower 4


done EXT Tower 9


ALL DONE


## 5  Results — per-tower R² (EC vs EXT) + RMSE/MAE/MBE

In [6]:
rows=[]
for (variant,t,model),scn in RESU.items():
    ov={k:np.nanmedian([scn[s][k] for s in SCENARIOS]) for k in ["R2","RMSE","MAE","MBE"]}
    rows.append({"variant":variant,"tower":t,"model":model,
                 **{f"{k}_overall":round(ov[k],3) for k in ov},
                 **{f"R2_{s}":round(scn[s]["R2"],3) for s in SCENARIOS}})
R=pd.DataFrame(rows)
piv=R.pivot_table(index=["tower","model"],columns="variant",values="R2_overall")
piv["dR2(EXT-EC)"]=(piv["EXT"]-piv["EC"]).round(3)
print("=== Overall median R2 (EC vs EXT) ===\n"+piv.to_string())
print("\n=== MBE (bias) overall, EC vs EXT — RFm pooled ===")
mb=R[R.model=="RFm_pool"].pivot_table(index="tower",columns="variant",values="MBE_overall")
print(mb.round(2).to_string())
print("\n=== RMSE overall (RFm pooled) ===")
print(R[R.model=="RFm_pool"].pivot_table(index="tower",columns="variant",values="RMSE_overall").round(1).to_string())
R.to_csv(RESULTS/"f08_summary.csv",index=False); print("\nsaved results/f08_summary.csv")

=== Overall median R2 (EC vs EXT) ===
variant            EC    EXT  dR2(EXT-EC)
tower model                              
2     MDS      -0.488 -0.310        0.178
      RFm_pool  0.478  0.490        0.012
      RFm_solo  0.395  0.385       -0.010
4     MDS      -0.236 -0.233        0.003
      RFm_pool  0.362  0.376        0.014
      RFm_solo  0.362  0.355       -0.007
9     MDS      -0.336 -0.320        0.016
      RFm_pool  0.350  0.364        0.014
      RFm_solo  0.337  0.339        0.002

=== MBE (bias) overall, EC vs EXT — RFm pooled ===
variant    EC   EXT
tower              
2        2.36  3.35
4        2.48  2.56
9        0.12  0.71

=== RMSE overall (RFm pooled) ===
variant     EC    EXT
tower                
2         96.6   95.9
4         97.9   98.7
9        120.7  120.0

saved results/f08_summary.csv


## 6  Append to benchmarks.csv (R2 + RMSE + MAE + MBE)

In [7]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="F-08"]
brows=[]
for (variant,t,model),scn in RESU.items():
    fs=f"{model}_{variant}"
    for sc in SCENARIOS:
        m=scn[sc]
        if pd.isna(m["R2"]) and pd.isna(m["RMSE"]): continue
        brows.append({"replication":"F-08","model":"MDS" if model=="MDS" else "RFm","tower":f"Tower {t}",
            "feature_set":fs,"scenario":sc,"split":"fullCV",
            "R2":round(float(m["R2"]),4) if pd.notna(m["R2"]) else np.nan,
            "RMSE":round(float(m["RMSE"]),4),"MAE":round(float(m["MAE"]),4),"MBE":round(float(m["MBE"]),4),
            "date":today,"notes":"F08 EC-vs-external sensor sourcing ("+variant+"); full-period gap-CV all towers (D-35)"})
new=pd.DataFrame(brows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} F-08 rows. Total {len(comb)}.")

Wrote 90 F-08 rows. Total 2963.
